# Influence of library diversity `d0` on signal fidelity and recovery, at fixed `mu=10`, `rho=1e-4`, `D=1e8`, `T_viab=0.8`, `noise_viab=0.5`

Every other notebook in this folder fixes the pool size `d0` and sweeps some other parameter.
This one does the opposite: `d0` (the number of distinct sequences in the library) IS the swept
variable, from `5,000` to `200,000`.

**Why this isolates a real trade-off, not just "more data is better".** `mu = rho * N1 / d0`
(mean HEK cells transfected per sequence) is a PER-SEQUENCE quantity -- so at every `d0`, `N1` is
recomputed as `N1 = mu * d0 / rho` to keep `mu=10` exactly fixed (same compensation trick as
`mu_HEK_multiplicity_sweep.ipynb`'s own `mu` grid). But `D` (NGS sequencing depth) is a TOTAL
budget shared across the WHOLE pool, and is held FIXED at `1e8` here, NOT compensated. That means
reads-per-sequence (`D/d0`) shrinks as the library gets more diverse, even though each sequence
still gets the same transfection multiplicity `mu`. This notebook asks: does spreading a fixed
sequencing budget over a more diverse library degrade signal fidelity and recovery, independent
of the transfection-multiplicity effect already characterized in `mu_HEK_multiplicity_sweep.ipynb`?
This is directly a lab-protocol question -- "how much library diversity can a fixed NGS budget
actually support" -- not just an abstract parameter sweep.

Unlike the other sweeps in this folder, `d0` can't be changed by mutating an existing `Protocol`
object in place (`lambda0`/`lambda0p`/.../`lambda4` are all allocated at `(d0,)` in `__init__`) --
each grid point constructs a FRESH pool and a FRESH `ProtocolV3`, at the cost of losing the
"same reused object, advancing PRNG key" independence trick used elsewhere (every fresh
`ProtocolV3` restarts its internal key at the same hardcoded `key(42)`); independence across grid
points instead comes from each point using a genuinely different, differently-sized `sequences`
pool.

`rho=1e-4` here is 10x smaller than the `RHO_REF=1e-3` baseline used elsewhere in this folder --
irrelevant to the outcome by itself (`mu_HEK_multiplicity_sweep.ipynb` section 3 already showed
different `(rho, N1)` splits at the same `mu` give statistically indistinguishable results), it
only matters here because it sets the scale of `N1` needed to reach `mu=10` at each `d0`.

**`N0` convention.** `N0` (initial library size) is set to `150 * N1` at every grid point (not a
fixed constant) -- a wet-lab constraint: `N1` (HEK cells transfected) must stay at least 150x
smaller than `N0` (plasmid copies in the prep), so the ratio can't be pushed down to make room
for a larger `N1`. Combined with `mu=10` and `rho=1e-4`, this is what caps `DIVERSITY_GRID` at
`d0=200,000` (`N1=2e10`, `N0=3e12`, comfortably under the `~5e12` practical ceiling for `N0`;
the old `mu=50` grid reached `d0=1,000,000` but pushed `N0` past `5e12` there, which is why
`mu` was lowered rather than just capping `d0` alone).

## 0. Setup

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Must run before the first `import jax` anywhere -- same convention as the sibling notebooks.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from flax import nnx
import optax

from sequence_classesV1 import *
from analysisV1 import *
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
from initialize_weights import load_F_viab_aav9_potts, load_J_viab_aav9_potts, NUM_AMINO_ACIDS, NUM_POSITIONS

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

## 1. Ground truth weights

Identical real AAV9 `F_viab`/`J_viab` and permuted `F_sel`/`J_sel` as the sibling notebooks
(`F_sel`/`J_sel` are constructed but never used below -- only the viability channel, `target1`,
is swept here). Unlike the sibling notebooks, there is no single fixed `sequences` pool here --
each `d0` in the grid gets its own, built inside the sweep loop (section 6).

In [ ]:
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
F_viab = load_F_viab_aav9_potts()
J_viab = load_J_viab_aav9_potts()

key_F, key_J = jax.random.split(jax.random.key(0), 2)
sigma_F = jax.random.permutation(key_F, NUM_AMINO_ACIDS)
F_sel   = F_viab[sigma_F, :]
sigma_J = jax.random.permutation(key_J, NUM_AMINO_ACIDS)
J_sel   = J_viab[:, :, sigma_J, :][:, :, :, sigma_J]

print(f"F_viab shape: {F_viab.shape}   J_viab shape: {J_viab.shape}")

## 2. Fixed 50,000-sequence evaluation pool (SAME key in every notebook)

A separate, FIXED `50,000`-sequence pool, generated once with a hardcoded key
(`EVAL_POOL_KEY_SEED=999`, `EVAL_POOL_SIZE=50_000` -- the exact same pair of values used in
`deeper_mlp/diversity_sweep_deeper_mlp.ipynb` and every other sweep notebook in this folder),
held out from training entirely. This matters MORE here than in the fixed-`d0` sweeps: at the
smallest `d0` grid points, this notebook's own in-sweep test fold can be too small for
`topk_recovery(..., k=1000)` to mean much (it silently clamps to the whole fold once
`d0/2 < 1000`). Every model trained at every `d0` gets evaluated on this SAME `50,000` sequences,
so top-1000 recovery becomes comparable across the WHOLE `d0` grid and across every other
notebook using the same fixed key, independent of how big or small the training pool itself is.

In [ ]:
EVAL_POOL_KEY_SEED = 999   # fixed across every notebook -- reuse this exact value for comparable numbers
EVAL_POOL_SIZE     = 50_000

def compute_score_array(seq, F, J, L=NUM_POSITIONS):
    """Noiseless ground-truth score, decoupled from any Protocol instance -- same formula as
    Protocol.compute_score, applied directly to an arbitrary sequence array."""
    scores = jnp.sum(F[seq, jnp.arange(L)], axis=1)
    for i in range(L):
        for j in range(i + 1, L):
            scores = scores + J[i, j, seq[:, i], seq[:, j]]
    return scores

eval_sequences = jax.random.randint(jax.random.key(EVAL_POOL_KEY_SEED),
                                     shape=(EVAL_POOL_SIZE, NUM_POSITIONS),
                                     minval=0, maxval=NUM_AMINO_ACIDS)
eval_viab_score = np.asarray(compute_score_array(eval_sequences, F_viab, J_viab))
X_eval = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[np.asarray(eval_sequences)].reshape(EVAL_POOL_SIZE, -1)

print(f"fixed evaluation pool: {EVAL_POOL_SIZE:,} sequences (key seed={EVAL_POOL_KEY_SEED})")
print(f"GT score range on eval pool: [{eval_viab_score.min():.2f}, {eval_viab_score.max():.2f}]")

## 3. Fixed `mu=10`, `rho=1e-4`, `D=1e8`, `T_viab=0.8`, `noise_viab=0.5`, and the `d0` grid

`DIVERSITY_GRID` spans `5,000` to `200,000` (capped there by the `N0=150*N1` constraint at
`mu=10`, see section 1) -- below and up to the `20,000`/`200,000` pool sizes used elsewhere in
this folder, so those notebooks' results sit inside this one's range for reference. `K_TOPK`
stays `1000` (this project's running standard), safely below even the smallest grid point's
`1,500`-sequence test fold (`5,000 * 0.3`).

In [ ]:
RHO_FIXED        = 1e-4
D_FIXED          = 1e8
T_VIAB_FIXED     = 0.8
NOISE_VIAB_FIXED = 0.5
MU_FIXED         = 10
K_TOPK           = 1000
eps              = 1.0

DIVERSITY_GRID = [5_000, 10_000, 20_000, 50_000, 100_000, 200_000]

print(f"mu={MU_FIXED}  rho={RHO_FIXED}  D={D_FIXED:.0e}  T_viab={T_VIAB_FIXED}  noise_viab={NOISE_VIAB_FIXED}")
print(f"diversity (d0) grid: {DIVERSITY_GRID}")
for d0_val in DIVERSITY_GRID:
    print(f"  d0={d0_val:>9,}  ->  N1={MU_FIXED * d0_val / RHO_FIXED:.3e}  "
          f"reads/sequence (D/d0) = {D_FIXED / d0_val:.1f}")

## 4. Simulate the SAME experimental configuration on the fixed eval pool

To get a genuine `GT<->protocol` recovery number on the fixed `50,000`-sequence eval pool
(section 2), run a SEPARATE, independent `ProtocolV3` simulation dedicated to it alone -- NOT
mixed into any `d0`-sized training pool (that would dilute `D`/`d0` and distort `mu` at every
sweep point). Same `mu=10`/`rho`/`D`/`T_viab`/`noise_viab` as the rest of this notebook (ALL
fixed, none depend on the training sweep's `d0`), just with `N1` recomputed for the eval pool's
own `d0=50,000`. Since nothing here depends on `d0_val`, this simulation runs ONCE -- `target1_eval`
is a fixed reference the whole sweep can be compared against.

In [ ]:
N1_eval = MU_FIXED * EVAL_POOL_SIZE / RHO_FIXED

protocol_eval = ProtocolV3(multinomialNGS=True, N0=N1_eval*150, N1=N1_eval,
        dilution_factor=10, sequences=eval_sequences, D=D_FIXED,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
        noise_viab=NOISE_VIAB_FIXED, noise_sel=0.5, T_sel=1, T_viab=T_VIAB_FIXED,
        )
protocol_eval._rho = float(RHO_FIXED)

_bio_row_eval, ngs_row_eval = protocol_eval.loop_DE()
lambda0p_eval, lambda2p_eval, _lambda3p_eval = (np.asarray(a) for a in ngs_row_eval)
target1_eval = np.log((lambda2p_eval + eps) / (lambda0p_eval + eps))

r_gt_protocol_eval   = pearson(eval_viab_score, target1_eval)
rec_gt_protocol_eval = topk_recovery(eval_viab_score, target1_eval, k=K_TOPK)

print(f"mu (eval pool) = {protocol_eval._rho * protocol_eval.N1 / EVAL_POOL_SIZE:.2f}")
print(f"GT <-> protocol on the fixed eval pool: r={r_gt_protocol_eval:.3f}  "
      f"top-{K_TOPK} recovery={100 * rec_gt_protocol_eval:.1f}%")

## 5. `ProfileMLP`: same architecture as the sibling notebooks, adaptive batch size

Identical `ProfileMLP` and warmup-cosine-decay AdamW / early-stopping training loop as the
sibling notebooks. `batch_size` here is chosen PER `d0` (`max(256, n_train // 50)`, targeting
~50 optimizer steps/epoch) instead of a single fixed value -- the train fold ranges from
`3,500` sequences (smallest `d0`) to `140,000` (largest), a 40x spread, and a single fixed batch
size would mean wildly different training regimes (too few steps/epoch at one end, absurdly slow
epochs at the other) that would confound the diversity effect being measured here.

In [ ]:
def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


class ProfileMLP(nnx.Module):
    """
    MLP over the one-hot encoded per-position sequence (L=7 positions x A=20 amino acids
    -> 140 indicator features) -> scalar score. Identical to the sibling notebooks' ProfileMLP
    (Linear + BatchNorm + Dropout + gelu).
    """

    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (128, 64),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x, train: bool, rngs: nnx.Rngs = None):
        x = self.linear1(x)
        self.batchnorm1.use_running_average = not train
        x = self.batchnorm1(x)
        x = nnx.gelu(x)
        x = self.dropout1(x, rngs=rngs) if train else x
        x = self.linear2(x)
        self.batchnorm2.use_running_average = not train
        x = self.batchnorm2(x)
        x = nnx.gelu(x)
        x = self.dropout2(x, rngs=rngs) if train else x
        x = self.linear3(x)
        return x[:, 0]

In [ ]:
@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    """
    x : (batch, L*A) one-hot encoded per-position amino-acid indicators
    y : (batch,) target log enrichment
    """
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    y_pred = model(x, train=False)
    return jnp.mean((y_pred - y) ** 2)


@nnx.jit
def predict_log_enrichment(model, x):
    return model(x, train=False)


def train_profile_mlp(X_train, y_train, X_val, y_val, hidden_dims=(128, 64),
                       dropout_rate=0.1, epochs=300, batch_size=256,
                       peak_lr=1e-3, final_lr=1e-5, weight_decay=0,
                       patience=20, seed=0, verbose=True):
    rngs = nnx.Rngs(seed)
    model = ProfileMLP(input_dim=X_train.shape[1], hidden_dims=hidden_dims,
                        dropout_rate=dropout_rate, rngs=rngs)

    n_train         = X_train.shape[0]
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps     = steps_per_epoch * epochs

    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr,
        warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9),
        end_value=final_lr,
    )
    optimizer = nnx.Optimizer(
        model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param
    )

    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val,   y_val   = jnp.asarray(X_val),   jnp.asarray(y_val)

    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs, best_epoch, final_epoch = float("inf"), None, 0, 0, 0

    for epoch in range(epochs):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm = jax.random.permutation(perm_key, n_train)
        Xs, ys = X_train[perm], y_train[perm]

        for start in range(0, n_train - n_train % batch_size, batch_size):
            xb = Xs[start:start + batch_size]
            yb = ys[start:start + batch_size]
            train_step(model, optimizer, xb, yb, rngs)

        val_loss = float(eval_step(model, X_val, y_val))
        final_epoch = epoch
        if val_loss < best_val - 1e-6:
            best_val, bad_epochs, best_epoch = val_loss, 0, epoch
            best_state = nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"  early stop at epoch {epoch}, best val MSE = {best_val:.4f}")
                break

    if best_state is not None:
        nnx.update(model, best_state)
    # best_epoch: epoch at which best_val was reached (how long it took to actually converge).
    # final_epoch: last epoch run before stopping (best_epoch + patience, unless it used all `epochs`).
    # steps_per_epoch * (final_epoch + 1) = total optimizer steps actually taken.
    return model, best_val, best_epoch, final_epoch, steps_per_epoch

## 6. Sweep: one fresh pool + one fresh `ProtocolV3` + one fresh `ProfileMLP` per `d0`

For each `d0`: draw a fresh `sequences` pool of that size (`jax.random.fold_in` off a shared base
key, so every `d0` gets an independent draw), build `N1 = mu * d0 / rho`, construct a fresh
`ProtocolV3`, run one `loop_DE()` round to get `target1`, split 50/50 into train/test, train a
`ProfileMLP` on the train fold (adaptive `batch_size`, section 3), then compare on the held-out
test fold: Pearson `r` and top-`K_TOPK` recovery, each across the three pairwise views (GT,
protocol, MLP).

In [ ]:
records = []
base_key = jax.random.key(1)

for d0_val in tqdm(DIVERSITY_GRID, desc="diversity (d0) sweep"):
    pool_key  = jax.random.fold_in(base_key, d0_val)
    sequences = jax.random.randint(pool_key, shape=(d0_val, NUM_POSITIONS),
                                    minval=0, maxval=NUM_AMINO_ACIDS)
    N1_val = MU_FIXED * d0_val / RHO_FIXED

    protocol = ProtocolV3(multinomialNGS=True, N0=N1_val*150, N1=N1_val,
            dilution_factor=10, sequences=sequences, D=D_FIXED,
            F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
            noise_viab=NOISE_VIAB_FIXED, noise_sel=0.5, T_sel=1, T_viab=T_VIAB_FIXED,
            )
    protocol._rho = float(RHO_FIXED)
    viab_score = np.array(protocol.compute_score(F_viab, J_viab))

    _bio_row, ngs_row = protocol.loop_DE()
    lambda0p, lambda2p, _lambda3p = (np.asarray(a) for a in ngs_row)
    target1 = np.log((lambda2p + eps) / (lambda0p + eps))

    X_all = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[np.asarray(sequences)].reshape(d0_val, -1)
    idx_train, idx_test = train_test_split(np.arange(d0_val), test_size=0.3, random_state=0)
    X_train_full, X_test   = X_all[idx_train], X_all[idx_test]
    viab_score_test        = viab_score[idx_test]
    target1_train, target1_test = target1[idx_train], target1[idx_test]

    Xtr, ytr, Xva, yva = split_train_val(X_train_full, target1_train, val_frac=0.15, seed=0)
    batch_size = max(256, len(Xtr) // 50)
    model, val_mse, best_epoch, final_epoch, steps_per_epoch = train_profile_mlp(
        Xtr, ytr, Xva, yva, epochs=300, patience=20, batch_size=batch_size, verbose=False)
    pred_test = np.asarray(predict_log_enrichment(model, jnp.asarray(X_test)))
    pred_eval = np.asarray(predict_log_enrichment(model, jnp.asarray(X_eval)))

    records.append(dict(
        d0=d0_val,
        batch_size=batch_size,
        val_mse=val_mse,
        best_epoch=best_epoch,          # epoch at which the best val MSE was reached
        n_epochs_run=final_epoch + 1,   # total epochs before early stop fired
        total_steps=steps_per_epoch * (final_epoch + 1),  # total optimizer updates actually taken
        r_gt_protocol=pearson(viab_score_test, target1_test),
        r_gt_mlp=pearson(viab_score_test, pred_test),
        r_protocol_mlp=pearson(target1_test, pred_test),
        rec_gt_mlp_eval=topk_recovery(eval_viab_score, pred_eval, k=K_TOPK),
        r_protocol_mlp_eval=pearson(target1_eval, pred_eval),
        rec_protocol_mlp_eval=topk_recovery(target1_eval, pred_eval, k=K_TOPK),
    ))

diversity_sweep_df = pd.DataFrame(records)
diversity_sweep_df

## 7. Pearson correlation vs `d0`

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(diversity_sweep_df["d0"], diversity_sweep_df["r_gt_protocol"],  "o-", label="GT <-> protocol")
ax.plot(diversity_sweep_df["d0"], diversity_sweep_df["r_gt_mlp"],       "o-", label="GT <-> MLP")
ax.plot(diversity_sweep_df["d0"], diversity_sweep_df["r_protocol_mlp"], "o-", label="protocol <-> MLP")
ax.set_xscale("log")
ax.set_ylim(0, 1)
ax.set_xlabel("d0 (library diversity -- number of distinct sequences)")
ax.set_ylabel("Pearson r (on the log enrichments)")
ax.set_title(f"Pearson r vs d0, mu={MU_FIXED}, rho={RHO_FIXED:g}, D={D_FIXED:.0e}, "
             f"T_viab={T_VIAB_FIXED}, noise_viab={NOISE_VIAB_FIXED} fixed")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## 8. Top-1000 recovery on the FIXED 50,000-sequence eval pool vs `d0`

Same `50,000` sequences (section 2) at every `d0` -- comparable across the whole grid AND across
every other sweep notebook using the same `EVAL_POOL_KEY_SEED=999`/`EVAL_POOL_SIZE=50_000` pair.
`GT<->protocol` (gray line) comes from section 4's dedicated eval-pool simulation -- constant
across `d0` (independent of the training sweep), the natural reference for whether the MLP
actually beats the raw protocol measurement on this common benchmark.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.axhline(100 * rec_gt_protocol_eval, color="gray", lw=1.5, label=f"GT <-> protocol (r={r_gt_protocol_eval:.3f})")
ax.plot(diversity_sweep_df["d0"], 100 * diversity_sweep_df["rec_gt_mlp_eval"], "o-", color="tab:red", label="GT <-> MLP (fixed eval pool)")
ax.axhline(100 * K_TOPK / EVAL_POOL_SIZE, color="black", lw=1, ls="--", label="random baseline")
ax.set_xscale("log")
ax.set_ylim(0, 100)
ax.set_xlabel("d0 (library diversity -- number of distinct sequences)")
ax.set_ylabel(f"Top-{K_TOPK} recovery on the fixed {EVAL_POOL_SIZE:,}-sequence eval pool (%)")
ax.set_title("GT <-> MLP recovery on a COMMON, fixed evaluation population")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Training diagnostics: does the model actually get more gradient updates as `d0` grows?

`batch_size = max(256, n_train // 50)` was chosen to keep ~50 steps/epoch (and wall-clock/epoch)
roughly constant across the whole grid -- which means a bigger `d0` gives the model bigger,
smoother batches, NOT necessarily more total gradient updates. If `best_epoch` (epoch the best
val MSE was reached at) stays roughly flat across `d0`, the model converges in about the same
number of update steps regardless of how much data it's given -- consistent with "the model is
too shallow to need more data, so growing `d0` doesn't buy it a better fit, only noisier labels
to fit against." If `best_epoch`/`total_steps` instead grow with `d0`, the model IS making use
of the extra data (taking longer, i.e. more updates, to converge on a better fit).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(diversity_sweep_df["d0"], diversity_sweep_df["best_epoch"], "o-", color="tab:blue")
ax.set_xscale("log")
ax.set_xlabel("d0")
ax.set_ylabel("epoch of best val MSE")
ax.set_title("How long training took to converge")
ax.grid(True, linestyle="--", alpha=0.3)

ax = axes[1]
ax.plot(diversity_sweep_df["d0"], diversity_sweep_df["total_steps"], "o-", color="tab:orange")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("d0")
ax.set_ylabel("total optimizer steps taken")
ax.set_title("Total gradient updates actually applied")
ax.grid(True, linestyle="--", alpha=0.3)

ax = axes[2]
ax.plot(diversity_sweep_df["d0"], diversity_sweep_df["val_mse"], "o-", color="tab:green")
ax.set_xscale("log")
ax.set_xlabel("d0")
ax.set_ylabel("best validation MSE")
ax.set_title("Does the model even fit its OWN training distribution better or worse?")
ax.grid(True, linestyle="--", alpha=0.3)

fig.tight_layout()
plt.show()

print(diversity_sweep_df[["d0", "batch_size", "best_epoch", "n_epochs_run", "total_steps", "val_mse"]])

## How to read this

- **Expected direction: DOWN as `d0` grows.** `mu` (transfection multiplicity per sequence) is
  deliberately held fixed by compensating `N1`, so it cannot explain any trend here -- what's
  NOT compensated is `D` (total sequencing budget), so reads-per-sequence (`D/d0`, printed in
  section 3) shrinks as the library gets more diverse. If both curves decline as `d0` grows,
  that's the fixed-NGS-budget dilution effect, isolated from the multiplicity effect
  `mu_HEK_multiplicity_sweep.ipynb` already covers.
- **`GT <-> protocol` (blue)** is the more "mechanical" signature of this dilution: as reads/sequence
  drops, the raw NGS-measured `target1` should get systematically noisier, the same way low `mu`
  or low `D` (at fixed `d0`, see `D_sequencing_depth_sweep.ipynb`) do.
- **`GT <-> MLP` (orange) vs `GT <-> protocol` (blue)**: at the largest `d0` (fewest reads/sequence,
  but also the LARGEST training set for the MLP), does the extra training data compensate for the
  thinner per-sequence signal -- i.e. does `GT<->MLP` decline more slowly than `GT<->protocol`, or
  even overtake it, the same denoising signature discussed in `noise_viab_sweep.ipynb`?
- **`protocol <-> MLP` (green)** should stay the lowest of the three at every `d0`, same reasoning
  as the other sweeps in this folder (the model has no information about the test fold's specific
  noise draw).
- **Section 9 checks the training-budget hypothesis directly**: if `best_epoch`/`total_steps` don't grow with `d0`, the model isn't actually exploiting the extra data any more at large `d0` than at small `d0` -- the adaptive `batch_size` (kept to hold steps/epoch roughly constant across the grid) is a likely reason why, and worth revisiting (e.g. a fixed, smaller `batch_size` so steps/epoch scales WITH `d0`) if this turns out to be the dominant explanation for the decline.
- **Top-1000 recovery (section 8) is ONLY computed on the fixed 50,000-sequence eval pool now**,
  not on the in-sweep test fold -- that fold's own size scales with `d0` (and with this
  notebook's own `test_size=0.3`, shrinks to just `1,500` sequences at the smallest grid point),
  which made `topk_recovery(k=1000)` either degenerate or simply not comparable across `d0`.
  Pearson `r` (section 7) still uses the in-sweep test fold, since `r` doesn't have this
  degeneracy problem -- only recovery does.